In [ ]:
!pip install scikit-learn pandas numpy --quiet
print('All libraries ready!')

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor, GradientBoostingClassifier, IsolationForest
from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.metrics import mean_absolute_error, classification_report

np.random.seed(42)
print('Imports done!')

**Premium Price Prediction**

In [ ]:
# ── Dummy Training Data ────────────────────────────────────────────────────
n = 500

premium_data = pd.DataFrame({
    'zone_flood_risk':             np.random.randint(1, 10, n),
    'avg_monthly_rain_mm':         np.random.randint(10, 400, n),
    'avg_aqi':                     np.random.randint(50, 350, n),
    'disruption_days_last_month':  np.random.randint(0, 10, n),
    'rider_active_hours_week':     np.random.randint(20, 70, n),
    'rider_tenure_months':         np.random.randint(1, 36, n),
    'city_tier':                   np.random.choice([1, 2, 3], n),
})

# Higher risk = higher premium
premium_data['weekly_premium'] = (
    49
    + premium_data['zone_flood_risk'] * 1.5
    + premium_data['avg_monthly_rain_mm'] * 0.03
    + premium_data['avg_aqi'] * 0.02
    + premium_data['disruption_days_last_month'] * 0.8
    - premium_data['rider_tenure_months'] * 0.2  # loyalty discount
    + np.random.normal(0, 1.5, n)
).clip(49, 79).round(2)

print('Sample Data:')
premium_data.head()

In [ ]:
# ── Train Model ────────────────────────────────────────────────────────────
X = premium_data.drop('weekly_premium', axis=1)
y = premium_data['weekly_premium']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

premium_model = RandomForestRegressor(n_estimators=100, random_state=42)
premium_model.fit(X_train, y_train)

preds = premium_model.predict(X_test)
print(f'Model trained! MAE: ₹{mean_absolute_error(y_test, preds):.2f}')

In [ ]:
# ── Predict for New Riders ─────────────────────────────────────────────────
def predict_premium(zone_flood_risk, avg_monthly_rain_mm, avg_aqi,
                    disruption_days_last_month, rider_active_hours_week,
                    rider_tenure_months, city_tier):
    inp = pd.DataFrame([{
        'zone_flood_risk': zone_flood_risk,
        'avg_monthly_rain_mm': avg_monthly_rain_mm,
        'avg_aqi': avg_aqi,
        'disruption_days_last_month': disruption_days_last_month,
        'rider_active_hours_week': rider_active_hours_week,
        'rider_tenure_months': rider_tenure_months,
        'city_tier': city_tier,
    }])
    return round(float(premium_model.predict(inp)[0]), 2)

# Arjun — Chennai, high flood risk zone
arjun = predict_premium(8, 320, 110, 5, 55, 6, 1)
print(f'Arjun (Chennai, high risk) weekly premium: ₹{arjun}')

# Priya — Coimbatore, low risk zone
priya = predict_premium(2, 60, 70, 1, 40, 18, 2)
print(f'Priya (Coimbatore, low risk) weekly premium: ₹{priya}')

**Risk Zone Scoring**

In [ ]:
# ── Dummy Zone Data ────────────────────────────────────────────────────────
n = 200

zones = pd.DataFrame({
    'zone_id':                 [f'ZONE_{i:03d}' for i in range(n)],
    'city':                    np.random.choice(['Chennai', 'Mumbai', 'Delhi', 'Bengaluru', 'Hyderabad'], n),
    'avg_rainfall_mm':         np.random.randint(10, 400, n),
    'flood_incidents_yearly':  np.random.randint(0, 15, n),
    'avg_aqi':                 np.random.randint(40, 350, n),
    'curfew_days_yearly':      np.random.randint(0, 10, n),
    'claims_last_6_months':    np.random.randint(0, 50, n),
})

print('Sample Zone Data:')
zones.head()

In [ ]:
# ── Train KMeans ───────────────────────────────────────────────────────────
zone_features = ['avg_rainfall_mm', 'flood_incidents_yearly',
                 'avg_aqi', 'curfew_days_yearly', 'claims_last_6_months']

zone_scaler = MinMaxScaler()
scaled = zone_scaler.fit_transform(zones[zone_features])

kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
zones['cluster'] = kmeans.fit_predict(scaled)

# Map clusters to risk scores
centers = kmeans.cluster_centers_.mean(axis=1)
rank = centers.argsort()
score_map = {rank[0]: 20, rank[1]: 55, rank[2]: 85}

zones['risk_score'] = (
    zones['cluster'].map(score_map) + np.random.randint(-10, 10, n)
).clip(0, 100)

zones['risk_label'] = zones['risk_score'].apply(
    lambda s: 'Low' if s < 40 else ('Medium' if s < 70 else 'High')
)

print('Zone risk distribution:')
print(zones['risk_label'].value_counts())
print()
zones[['zone_id', 'city', 'risk_score', 'risk_label']].head(10)

In [ ]:
# ── Score a New Zone ───────────────────────────────────────────────────────
def score_zone(avg_rainfall_mm, flood_incidents_yearly,
               avg_aqi, curfew_days_yearly, claims_last_6_months):
    inp = np.array([[avg_rainfall_mm, flood_incidents_yearly,
                     avg_aqi, curfew_days_yearly, claims_last_6_months]])
    scaled_inp = zone_scaler.transform(inp)
    cluster = kmeans.predict(scaled_inp)[0]
    score = int(np.clip(score_map[cluster] + np.random.randint(-5, 5), 0, 100))
    label = 'Low' if score < 40 else ('Medium' if score < 70 else 'High')
    return {'risk_score': score, 'risk_label': label}

# T. Nagar Chennai — flood prone
print('T. Nagar Chennai:', score_zone(350, 12, 130, 3, 40))

# Coimbatore — safe zone
print('Coimbatore:      ', score_zone(55, 1, 65, 0, 5))

** Fraud Detection**

In [ ]:
# ── Dummy Claims Data ──────────────────────────────────────────────────────
n = 300

claims = pd.DataFrame({
    'claim_id':                       [f'CLM_{i:04d}' for i in range(n)],
    'gps_zone_match':                 np.random.choice([True, False], n, p=[0.85, 0.15]),
    'deliveries_before_event':        np.random.randint(0, 15, n),
    'peers_claimed_same_event':       np.random.randint(0, 80, n),
    'claims_this_week':               np.random.randint(1, 5, n),
    'avg_weekly_claims_historical':   np.random.uniform(0.5, 2.5, n),
    'time_since_policy_days':         np.random.randint(1, 200, n),
    'claim_amount':                   np.random.randint(400, 1600, n),
})

# Inject fraudulent claims
fraud_idx = np.random.choice(n, 40, replace=False)
claims.loc[fraud_idx, 'gps_zone_match'] = False
claims.loc[fraud_idx, 'deliveries_before_event'] = 0
claims.loc[fraud_idx, 'peers_claimed_same_event'] = np.random.randint(0, 3, 40)
claims.loc[fraud_idx, 'claims_this_week'] = np.random.randint(3, 5, 40)

print('Sample Claims Data:')
claims.head()

In [ ]:
# ── Train Isolation Forest ─────────────────────────────────────────────────
fraud_features = [
    'deliveries_before_event', 'peers_claimed_same_event',
    'claims_this_week', 'avg_weekly_claims_historical',
    'claim_amount', 'time_since_policy_days'
]

fraud_scaler = StandardScaler()
X_fraud = fraud_scaler.fit_transform(claims[fraud_features])

iso_forest = IsolationForest(contamination=0.1, random_state=42)
claims['anomaly'] = iso_forest.fit_predict(X_fraud)
claims['ml_flagged'] = claims['anomaly'] == -1

print('Isolation Forest trained!')
print(f'ML flagged {claims["ml_flagged"].sum()} suspicious claims out of {n}')

In [ ]:
# ── Check a Claim ──────────────────────────────────────────────────────────
def check_claim(gps_zone_match, deliveries_before_event, peers_claimed_same_event,
                claims_this_week, avg_weekly_claims_historical,
                claim_amount, time_since_policy_days):

    # Rule checks
    flags = []
    if not gps_zone_match:
        flags.append('GPS zone mismatch')
    if deliveries_before_event == 0 and peers_claimed_same_event < 5:
        flags.append('No activity + very few peers claimed')
    if claims_this_week > 2:
        flags.append('Too many claims this week')
    if time_since_policy_days < 3:
        flags.append('Policy too new — possible fraud')

    # ML check
    inp = fraud_scaler.transform([[deliveries_before_event, peers_claimed_same_event,
                                    claims_this_week, avg_weekly_claims_historical,
                                    claim_amount, time_since_policy_days]])
    ml_flag = iso_forest.predict(inp)[0] == -1

    # Final decision
    if flags and ml_flag:
        decision = '🔴 REJECT'
    elif flags or ml_flag:
        decision = '🟡 REVIEW'
    else:
        decision = '🟢 APPROVE'

    return {'decision': decision, 'flags': flags, 'ml_flagged': ml_flag}

# Genuine claim — heavy rain, 52 peers also claimed
print('Genuine claim:')
print(check_claim(True, 8, 52, 1, 0.8, 800, 45))

print()

# Fraudulent claim — GPS mismatch, no peers
print('Suspicious claim:')
print(check_claim(False, 0, 2, 3, 0.5, 1600, 2))

**Disruption Forecasting**

In [ ]:
# ── Dummy Historical Weekly Data ───────────────────────────────────────────
n = 600

forecast_data = pd.DataFrame({
    'avg_rainfall_forecast':              np.random.randint(0, 400, n),
    'max_temp_forecast':                  np.random.randint(25, 48, n),
    'min_aqi_forecast':                   np.random.randint(30, 350, n),
    'strike_alert':                       np.random.choice([0, 1], n, p=[0.92, 0.08]),
    'flood_alert':                        np.random.choice([0, 1], n, p=[0.88, 0.12]),
    'historical_disruptions_this_month':  np.random.randint(0, 8, n),
})

# Label: disruption will happen if any threshold likely breached
forecast_data['disruption_next_week'] = (
    (forecast_data['avg_rainfall_forecast'] > 150) |
    (forecast_data['max_temp_forecast'] > 42) |
    (forecast_data['min_aqi_forecast'] > 280) |
    (forecast_data['strike_alert'] == 1) |
    (forecast_data['flood_alert'] == 1)
).astype(int)

print(f'Disruption rate in training data: {forecast_data["disruption_next_week"].mean():.1%}')
forecast_data.head()

In [ ]:
# ── Train Model ────────────────────────────────────────────────────────────
forecast_features = [
    'avg_rainfall_forecast', 'max_temp_forecast', 'min_aqi_forecast',
    'strike_alert', 'flood_alert', 'historical_disruptions_this_month'
]

X = forecast_data[forecast_features]
y = forecast_data['disruption_next_week']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

forecast_model = GradientBoostingClassifier(n_estimators=100, random_state=42)
forecast_model.fit(X_train, y_train)

print('✅ Model trained!')
print(classification_report(y_test, forecast_model.predict(X_test)))

In [ ]:
# ── Admin Dashboard: All Cities Next Week ─────────────────────────────────
cities = pd.DataFrame({
    'zone':                              ['Chennai North', 'Mumbai Dharavi', 'Delhi Rohini',
                                          'Bengaluru Koramangala', 'Hyderabad Hitech City'],
    'avg_rainfall_forecast':             [280, 60, 20, 90, 40],
    'max_temp_forecast':                 [34, 33, 44, 32, 38],
    'min_aqi_forecast':                  [120, 160, 310, 90, 130],
    'strike_alert':                      [0, 1, 0, 0, 0],
    'flood_alert':                       [1, 0, 0, 0, 0],
    'historical_disruptions_this_month': [5, 3, 2, 1, 2],
})

probs = forecast_model.predict_proba(cities[forecast_features])[:, 1]
cities['disruption_probability'] = probs.round(2)
cities['risk_level'] = cities['disruption_probability'].apply(
    lambda p: '🔴 High' if p >= 0.70 else ('🟡 Medium' if p >= 0.35 else '🟢 Low')
)

print('\n📊 Admin Dashboard — Next Week Disruption Forecast')
print('=' * 55)
cities[['zone', 'disruption_probability', 'risk_level']]